# Домашнее задание. Линейная регрессия по точной формуле

Ты работаешь в сервисе объявлений о недвижимости. Нужно сделать модель, которая оценивает цену квартиры по её характеристикам.

План работы:

1. Сгенерировать датасет и посмотреть на него.
2. Заполнить пропуски средним значением столбца.
3. Сделать one-hot encoding категориального признака.
4. Собрать матрицу признаков $X$ и вектор ответов $y$.
5. Найти веса модели по точной формуле $w = (X^T X)^{-1} X^T y$.
6. Предсказать цену новых квартир и сравнить с похожими квартирами из обучающих данных.

Пиши код под комментариями в каждой ячейке. Нужны только `numpy` и `pandas`.

## Шаг 0. Генерация датасета

Просто запусти ячейку, менять ничего не нужно.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 200

district = rng.choice(["Центр", "Спальный", "Окраина"], size=n, p=[0.3, 0.45, 0.25])
area = np.round(rng.uniform(25, 110, n), 1)
floor = rng.integers(1, 26, n)
metro_km = np.round(rng.gamma(2, 0.8, n) + np.where(district == "Окраина", 1.5, 0), 1)
district_bonus = pd.Series(district).map({"Центр": 4.0, "Спальный": 1.5, "Окраина": 0.0}).values
price = 2 + 0.17 * area - 0.5 * metro_km + 0.04 * floor + district_bonus + rng.normal(0, 0.8, n)

flats = pd.DataFrame({
    "area_m2": area,
    "floor": floor.astype(float),
    "metro_km": metro_km,
    "district": district,
    "price_mln": np.round(price, 2),
})

# в реальных данных часть значений не заполнена
for col, share in [("area_m2", 0.06), ("floor", 0.08), ("metro_km", 0.10)]:
    flats.loc[rng.random(n) < share, col] = np.nan

flats.head(10)

**Описание столбцов**

| столбец | описание |
|---|---|
| `area_m2` | площадь квартиры, м² |
| `floor` | этаж |
| `metro_km` | расстояние до ближайшего метро, км |
| `district` | тип района: Центр, Спальный, Окраина |
| `price_mln` | цена квартиры, млн руб. — **целевая переменная** |

## Шаг 1. Смотрим на данные

In [ ]:
# Выведи размер таблицы (flats.shape)
# Выведи число пропусков в каждом столбце (flats.isna().sum())
# Выведи, сколько квартир в каждом районе (flats['district'].value_counts())

## Шаг 2. Заполняем пропуски

Модель не умеет работать с пропусками: формула с матрицами не посчитается, если в $X$ есть `NaN`. Заполним каждый пропуск **средним значением своего столбца**. Среднее считается только по заполненным значениям — `pandas` по умолчанию пропуски при подсчёте среднего пропускает.

In [ ]:
# Для каждого столбца с пропусками (area_m2, floor, metro_km):
#     посчитай среднее этого столбца: flats[col].mean()
#     выведи его
#     заполни пропуски этим средним: flats[col] = flats[col].fillna(среднее)
# Проверь, что пропусков больше нет: flats.isna().sum()

## Шаг 3. One-hot encoding

Признак `district` — текст, а в формулу можно подставлять только числа. Превратим его в столбцы из 0 и 1.

У района 3 варианта, но столбцов нужно **N − 1 = 2**. Третий вариант кодируется так: в обоих столбцах нули. Если оставить все 3 столбца, их сумма в каждой строке будет равна 1 — то есть столбцу из единиц для свободного члена $w_0$. Тогда матрица $X^T X$ станет вырожденной и обратную к ней посчитать не получится.

| district | district_Спальный | district_Центр |
|---|---|---|
| Окраина | 0 | 0 |
| Спальный | 1 | 0 |
| Центр | 0 | 1 |

Здесь «Окраина» — базовый вариант: веса двух новых столбцов покажут, на сколько дороже Спальный район и Центр по сравнению с Окраиной.

In [ ]:
# Сделай one-hot encoding: pd.get_dummies(flats, columns=['district'], drop_first=True, dtype=int)
#     drop_first=True как раз оставляет N - 1 столбцов (по алфавиту первой идёт «Окраина», её и уберут)
# Сохрани результат в flats_encoded и выведи первые 5 строк
# Убедись, что вместо district появились два столбца district_Спальный и district_Центр

## Шаг 4. Собираем X и y

- $y$ — вектор цен (столбец `price_mln`).
- $X$ — матрица признаков: все остальные столбцы **плюс столбец из единиц в начале**. Столбец из единиц нужен для свободного члена $w_0$: тогда $\hat{y} = w_0 \cdot 1 + w_1 x_1 + \ldots$

Порядок столбцов в $X$: `const`, `area_m2`, `floor`, `metro_km`, `district_Спальный`, `district_Центр`.

In [ ]:
# Запиши список признаков: feature_cols = ['area_m2', 'floor', 'metro_km', 'district_Спальный', 'district_Центр']
# Получи y как numpy-массив: flats_encoded['price_mln'].to_numpy()
# Получи матрицу признаков: flats_encoded[feature_cols].to_numpy()
# Добавь слева столбец из единиц: np.column_stack([np.ones(len(y)), матрица])
# Сохрани результат в X и выведи X.shape и y.shape — должно быть (200, 6) и (200,)
# Выведи первые 3 строки X

## Шаг 5. Находим веса по точной формуле

Веса линейной регрессии, при которых сумма квадратов ошибок минимальна, считаются по формуле:

$$w = (X^T X)^{-1} X^T y$$

В numpy:

- транспонирование — `X.T`;
- матричное умножение — оператор `@` (не `*`: звёздочка умножает поэлементно);
- обратная матрица — `np.linalg.inv(...)`.

In [ ]:
# Посчитай XtX = X.T @ X и выведи его размер (должен быть 6 x 6)
# Посчитай обратную матрицу: np.linalg.inv(XtX)
# Посчитай w = обратная @ X.T @ y
# Выведи веса вместе с названиями: pd.Series(w, index=['const'] + feature_cols)

**Вопросы:**

1. На сколько, по мнению модели, вырастает цена при увеличении площади на 1 м²?
2. Что означает вес у `metro_km`? Почему он отрицательный?
3. На сколько квартира в Центре дороже такой же квартиры на Окраине? А в Спальном районе?

**Твой ответ:**

_..._

In [ ]:
# Проверка: посчитай предсказания для обучающих квартир y_pred = X @ w
# Посчитай среднюю абсолютную ошибку: np.abs(y - y_pred).mean()
# Выведи её — на сколько млн руб. модель в среднем ошибается на обучающих данных?

## Шаг 6. Предсказываем цену новых квартир

На сайт пришли пять новых объявлений. Цены у них нет — её нужно оценить моделью.

In [ ]:
# Просто запусти ячейку
test = pd.DataFrame({
    "area_m2":  [45.0, 80.0, 32.0, 60.0, 100.0],
    "floor":    [5,    12,   2,    20,   8],
    "metro_km": [0.8,  1.5,  3.5,  1.0,  4.0],
    "district": ["Центр", "Спальный", "Окраина", "Спальный", "Окраина"],
})
test

In [ ]:
# Сделай для test такой же one-hot encoding, как для обучающих данных
#     (подсказка: можно вручную: test['district_Спальный'] = (test['district'] == 'Спальный').astype(int),
#     и так же для Центра)
# Собери X_test так же, как X: столбец из единиц + столбцы feature_cols в том же порядке
# Посчитай предсказания: test_pred = X_test @ w
# Добавь их в таблицу test как столбец price_pred и выведи test

**Вопрос:** почему у тестовых квартир нужно делать one-hot encoding в точности так же, как у обучающих, и столбцы ставить в том же порядке? Что будет, если перепутать порядок?

**Твой ответ:**

_..._

## Шаг 7. Справедлива ли цена?

Проверим предсказания здравым смыслом: найдём в обучающих данных квартиры, **похожие** на каждую новую, и посмотрим, сколько они стоят.

Похожими будем считать квартиры из того же района, площадь которых отличается не больше чем на 10 м², а расстояние до метро — не больше чем на 1 км.

In [ ]:
# Для каждой квартиры из test:
#     отбери из flats квартиры того же района, у которых |area_m2 - площадь| <= 10 и |metro_km - метро| <= 1
#     посчитай, сколько таких квартир нашлось, и их среднюю цену
# Добавь в test столбцы similar_count и similar_mean_price
# Выведи test со столбцами district, area_m2, metro_km, price_pred, similar_count, similar_mean_price

**Вопросы:**

1. Близки ли предсказания модели к средним ценам похожих квартир?
2. Если для какой-то квартиры похожих нашлось мало или не нашлось совсем — почему так и можно ли доверять сравнению?
3. Похожие квартиры отличаются этажом и немного площадью. Почему модель может оценить цену точнее, чем просто среднее по похожим?

**Твой ответ:**

_..._